<a href="https://colab.research.google.com/github/sujithkumarmp/cloud-ai-basics/blob/main/hf_diffuser_video_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision torchaudio diffusers transformers accelerate

In [2]:
import accelerate
import torch
import diffusers
import transformers

In [3]:
pip install opencv-python

In [4]:
import torch
from diffusers import DiffusionPipeline

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [5]:
pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-video-diffusion-img2vid-xt", device_map="cuda")
pipe.to(torch.bfloat16)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

StableVideoDiffusionPipeline {
  "_class_name": "StableVideoDiffusionPipeline",
  "_diffusers_version": "0.37.1",
  "_name_or_path": "stabilityai/stable-video-diffusion-img2vid-xt",
  "feature_extractor": [
    "transformers",
    "CLIPImageProcessor"
  ],
  "image_encoder": [
    "transformers",
    "CLIPVisionModelWithProjection"
  ],
  "scheduler": [
    "diffusers",
    "EulerDiscreteScheduler"
  ],
  "unet": [
    "diffusers",
    "UNetSpatioTemporalConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLTemporalDecoder"
  ]
}

In [6]:
pipe.enable_model_cpu_offload()

In [7]:
from diffusers.utils import load_image, export_to_video

In [8]:
prompt = "A man with short gray hair plays a red electric guitar."
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png"
)

In [9]:
image.size

(928, 512)

In [10]:
generator = torch.manual_seed(42)

In [25]:
import torch
import gc

# 1. Clear memory
gc.collect()
torch.cuda.empty_cache()

# 2. Re-cast the entire pipeline to float16 to avoid BFloat16/Float32 mismatches
pipe.to(dtype=torch.float16)

# 3. Use 144p resolution
new_width, new_height = 256, 144
small_image = image.resize((new_width, new_height))

# 4. Aggressive memory optimizations
pipe.enable_sequential_cpu_offload()
pipe.enable_attention_slicing()

try:
    # Run with float16 logic
    output = pipe(
        small_image,
        width=new_width,
        height=new_height,
        num_frames=14,
        decode_chunk_size=1,
        generator=generator
    )
    frames = output.frames
    print(f"Successfully generated {len(frames)} frames.")
except Exception as e:
    print(f"Error: {e}")

  0%|          | 0/25 [00:00<?, ?it/s]

Successfully generated 1 frames.


In [26]:
# Export the generated frames to a video file
export_to_video(frames[0], "generated_video.mp4", fps=7)
print("Video saved as generated_video.mp4")

Video saved as generated_video.mp4
